# Automatic Values in Python Enumerations — Advanced Tutorial, Volume II

## Learn by predicting, running, explaining, and testing

This is a **new set of worked problems**, not a reformatting of the previous problem workbook. We follow the supplied tutorial's rhythm: start with a small motivating question, break it into logical experiments, add explanatory **Markdown cells between code cells**, then finish each problem with tests and a principle worth remembering.

**Scope:** `enum.auto()`, `_generate_next_value_(name, start, count, last_values)`, aliasing, `@unique`, reusable bases, and the consequences of treating generated values as a public interface. Further material develops those same ideas through multi-field members, deterministic identifiers, data migrations, flags, and a realistic capstone.

**Environment:** standard library only; Python **3.13+** recommended for the version-specific tests. The baseline demonstrations work on many earlier versions, but the explicitly version-gated sections explain the differences. Run cells **top to bottom**. A deliberate failure is always caught and explained: `Run All` should succeed.

**Teaching convention:** all enum member names are `UPPER_CASE`; a generated value is an *implementation detail* unless we explicitly make it part of a documented external contract.

## Road map

1. Find out exactly *when* automatic values are generated.
2. Allocate bounded codes while skipping reserved numbers.
3. Design intentional aliases rather than accidentally creating them.
4. Combine `auto()` with additional per-member metadata.
5. Investigate why placing `auto()` inside a container is different.
6. Diagnose a generator defined in the wrong place.
7. Reuse a value-generation policy across independent enums.
8. Derive stable identifiers from names, and audit collision risks.
9. Support old user inputs without adding accidental aliases.
10. Allocate sparse `Flag` bits for forward compatibility.
11. Understand equality-based aliasing with structured values.
12. Fail safely when the available ID namespace is exhausted.
13. Upgrade stored member names between two software versions.
14. Build a small event protocol that integrates the entire lesson.

Each problem includes an initial question, staged investigation, annotated answer, and assertions. After the capstone we run a cumulative audit.

## Version and reference notes

Your source tutorial shows the default sequential behavior and demonstrates that manually assigned values may produce aliases. It also emphasizes `_generate_next_value_`, all four arguments, reusable generator-only parent classes, and the risk of depending on auto-assigned values. We keep those foundations, but all the challenges below are **new scenarios**.

Python's documented rules evolve: for Python 3.13+, the standard `Enum`/`IntEnum` generator selects a value above the *highest* preceding value and rejects incompatible value types; `Flag` generates powers of two; `StrEnum` generates a lowercase name. Top-level tuple assignments can resolve `auto()` (Python 3.11.1+), while nested list elements do not. Our important generators **override the method explicitly** to make their own behavior stable and testable.

Official references: [enum library](https://docs.python.org/3/library/enum.html) and [Enum HOWTO](https://docs.python.org/3/howto/enum.html). Consult the documentation for the actual interpreter when comparing behavior across versions.

In [1]:
import enum
import hashlib
import json
import sys
from dataclasses import dataclass
from enum import Enum, Flag, auto, unique

print('Python:', sys.version.split()[0])
print('Notebook examples use only the Python standard library.')
assert sys.version_info >= (3, 11), 'Use Python 3.11+ for the tuple auto() examples.'

Python: 3.13.7
Notebook examples use only the Python standard library.


---

# Problem 01 — A generator runs during class creation

**The situation.** A colleague assumes each access to `Phase.READY` calls the generator again.

**Your challenge.** Use a visible trace to establish the generator call order and whether enum access is lazy.

**Before running anything, predict:** How many generator calls happen for three `auto()` declarations?

We will build the solution a little at a time, inspect the surprising behavior, and finish with executable checks.

### Step 1 — Collect a trace outside the enum

A trace list is convenient here because it keeps output independent of the `Enum` display formatting. We record the name, number of earlier declarations, and all earlier *raw member values*. The method must appear before the members.

In [2]:
creation_log = []

class Phase(Enum):
    def _generate_next_value_(name, start, count, last_values):
        creation_log.append((name, count, tuple(last_values)))
        return 70 + count

    QUEUED = auto()
    ACTIVE = auto()
    FINISHED = auto()

for item in creation_log:
    print(item)

('QUEUED', 0, ())
('ACTIVE', 1, (70,))
('FINISHED', 2, (70, 71))


**What did we learn?** The callbacks already happened while Python executed the class statement. `last_values` grows after each declaration; the method is not invoked when a value is later read.

### Step 2 — Try reading the same member repeatedly

In [3]:
before = len(creation_log)
for _ in range(5):
    assert Phase.ACTIVE is Phase.ACTIVE
    assert Phase.ACTIVE.value == 71
print('Calls during member reads:', len(creation_log) - before)

Calls during member reads: 0


**Checkpoint / reasoning.** Zero new calls confirms that `auto()` does not generate a fresh value per access. Generated values belong to enum members, which are created once when the class is created.

### Step 3 — Use the trace as an executable specification

In [4]:
assert creation_log == [
    ('QUEUED', 0, ()),
    ('ACTIVE', 1, (70,)),
    ('FINISHED', 2, (70, 71)),
]
assert [item.value for item in Phase] == [70, 71, 72]
print('Problem 01: all assertions passed')

Problem 01: all assertions passed


**Solution in one sentence.** Think of `auto()` as a **class-definition-time instruction**, not a deferred value factory.

---

# Problem 02 — A reserved-range allocator

**The situation.** A service has numeric event IDs between 101 and 109, but codes 104 and 108 are reserved.

**Your challenge.** Implement automatic allocation that skips the reserved values while respecting manually assigned high-water marks.

**Before running anything, predict:** If `CHECKPOINT=105` is manual, what should the next generated code be?

We will build the solution a little at a time, inspect the surprising behavior, and finish with executable checks.

### Step 1 — Write down the invariant before coding

Our invariant is: every generated code must be **strictly larger** than all codes declared earlier, must not be reserved, and must not exceed 109. We use `last_values` rather than assuming `count` equals the desired numeric code: manual assignments and aliases can make the two diverge.

In [5]:
RESERVED_CODES = frozenset({104, 108})

class EventCode(Enum):
    def _generate_next_value_(name, start, count, last_values):
        candidate = max((100, *last_values)) + 1
        while candidate in RESERVED_CODES:
            candidate += 1
        if candidate > 109:
            raise ValueError(f'no available code for {name}')
        return candidate

    CREATE = auto()
    VERIFY = auto()
    CHECKPOINT = 105  # The external system has already assigned this code.
    RELEASE = auto()
    ARCHIVE = auto()
    CLOSE = auto()

print([(member.name, member.value) for member in EventCode])

[('CREATE', 101), ('VERIFY', 102), ('CHECKPOINT', 105), ('RELEASE', 106), ('ARCHIVE', 107), ('CLOSE', 109)]


**What did we learn?** `CREATE=101`, `VERIFY=102`, then the explicit checkpoint moves the high-water mark to 105. New generated values are 106, 107, and 109; 108 is skipped. We intentionally do **not** assume that numbers will be consecutive.

### Step 2 — Turn the design constraints into tests

In [6]:
event_values = [member.value for member in EventCode]
assert event_values == [101, 102, 105, 106, 107, 109]
assert not RESERVED_CODES.intersection(event_values)
assert event_values == sorted(set(event_values))
assert all(101 <= value <= 109 for value in event_values)
print('Valid allocated codes:', event_values)

Valid allocated codes: [101, 102, 105, 106, 107, 109]


### Step 3 — Consider a schema change

Adding a newly generated member **before** an existing member can still shift the later numbers. Skipping reserved codes prevents collisions with those reservations; it does not make automatically allocated codes stable across code edits. Any code published to another system should normally be assigned explicitly and versioned.

**Checkpoint / reasoning.** Why did we not use `100 + count + 1`? Because after the manually assigned 105, the count is not the high-water mark. The rule would potentially assign earlier values or violate the intended ordering.

**Solution in one sentence.** Specify the numeric invariant, calculate from the actual previous values, and test reserved and exhaustion boundaries.

---

# Problem 03 — Intentional compatibility aliases

**The situation.** A package accepts both current and legacy spellings of internal status constants.

**Your challenge.** Use the generator to assign the *same value* to a `_LEGACY` name that immediately follows its canonical member, without relying on silent accidental collisions.

**Before running anything, predict:** How many members will iteration yield after four names are declared?

We will build the solution a little at a time, inspect the surprising behavior, and finish with executable checks.

### Step 1 — Encode the naming rule explicitly

Here the alias is intentional. An `_LEGACY` declaration must reuse the immediately preceding value; any other declaration should obtain a new one. This rule relies on **declaration order**, so we must document and test that constraint.

In [7]:
class JobStage(Enum):
    def _generate_next_value_(name, start, count, last_values):
        if name.endswith('_LEGACY'):
            if not last_values:
                raise ValueError('a legacy alias needs a preceding member')
            return last_values[-1]
        return max((40, *last_values)) + 1

    PREPARE = auto()
    PREPARE_LEGACY = auto()
    DELIVER = auto()
    DELIVER_LEGACY = auto()

print('Iteration:', list(JobStage))
print('Declared names:', list(JobStage.__members__))

Iteration: [<JobStage.PREPARE: 41>, <JobStage.DELIVER: 42>]
Declared names: ['PREPARE', 'PREPARE_LEGACY', 'DELIVER', 'DELIVER_LEGACY']


### Step 2 — Why iteration loses names

An alias is not a distinct value-bearing enum object: `PREPARE_LEGACY is PREPARE`. Ordinary enum iteration yields canonical members only; `__members__` includes *every declared name*. This distinction matters for serializers, audits, and documentation generators.

In [8]:
assert JobStage.PREPARE_LEGACY is JobStage.PREPARE
assert JobStage.DELIVER_LEGACY is JobStage.DELIVER
assert len(list(JobStage)) == 2
assert len(JobStage.__members__) == 4
print({name: member.name for name, member in JobStage.__members__.items()})

{'PREPARE': 'PREPARE', 'PREPARE_LEGACY': 'PREPARE', 'DELIVER': 'DELIVER', 'DELIVER_LEGACY': 'DELIVER'}


### Step 3 — Distinguish intentional aliases from `@unique`

`@unique` rejects aliases regardless of intent. Therefore apply it only when the specification forbids aliases; do **not** silence the error if compatibility aliases are part of the design.

In [9]:
try:
    unique(JobStage)
except ValueError as exc:
    print('Expected @unique failure:', exc)
else:
    raise AssertionError('@unique must reject the intentional aliases')

Expected @unique failure: duplicate values found in <enum 'JobStage'>: PREPARE_LEGACY -> PREPARE, DELIVER_LEGACY -> DELIVER


### Step 4 — Audit the alias convention

In [10]:
for declared_name, member in JobStage.__members__.items():
    is_alias = declared_name != member.name
    assert is_alias == declared_name.endswith('_LEGACY')
print('Problem 03: aliases match the documented convention')

Problem 03: aliases match the documented convention


**Solution in one sentence.** Aliases are valid when deliberate; use `__members__` to audit them and reserve `@unique` for strictly alias-free classes.

---

# Problem 04 — An automatically assigned code plus route metadata

**The situation.** A routing table needs a stable numeric lookup key and a human-readable URL path per member.

**Your challenge.** Use `auto()` inside a top-level tuple and a custom `__new__` to separate the numeric member value from its metadata.

**Before running anything, predict:** Will `Route(301)` work if the member was declared with two fields?

We will build the solution a little at a time, inspect the surprising behavior, and finish with executable checks.

### Step 1 — Understand the two separate jobs

`_generate_next_value_` creates the **automatic code**. `__new__` receives the resolved tuple components, constructs the member, sets `_value_` to the integer, and stores the route path separately. If we fail to set `_value_` correctly, by-value lookup may not behave as intended.

In [11]:
class Route(Enum):
    def _generate_next_value_(name, start, count, last_values):
        return 300 + count

    def __new__(cls, code, path):
        member = object.__new__(cls)
        member._value_ = code
        member.path = path
        return member

    HOME = auto(), '/'
    SEARCH = auto(), '/search'
    SETTINGS = auto(), '/settings'

for member in Route:
    print(member.name, member.value, member.path)

HOME 300 /
SEARCH 301 /search
SETTINGS 302 /settings


**What did we learn?** `auto()` supplied 300, 301, and 302; the second tuple field became `.path`, **not** part of `.value`. This example assumes the top-level tuple behavior documented in Python 3.11.1+.

### Step 2 — Verify that enum lookup sees only the primary value

In [12]:
assert Route.SEARCH.value == 301
assert Route.SEARCH.path == '/search'
assert Route(301) is Route.SEARCH
assert Route['SEARCH'] is Route.SEARCH
print('By-value and by-name lookup agree')

By-value and by-name lookup agree


### Step 3 — Validate the metadata independently

Enum value uniqueness checks do not validate other attributes. If two routes share a path, `@unique` will not notice as long as their numeric values are distinct. Write a specific validation function for the path field.

In [13]:
def audit_paths(route_type):
    paths = [member.path for member in route_type]
    if not all(path.startswith('/') for path in paths):
        raise ValueError('every path must start with /')
    if len(paths) != len(set(paths)):
        raise ValueError('route paths must be unique')
    return len(paths)

assert audit_paths(Route) == 3
print('Metadata audit passed')

Metadata audit passed


**Checkpoint / reasoning.** The generator is deliberately based on `count`, not on the structure of `last_values`: in multi-field enum creation those historical assignment values need not have the same shape as a plain integer-only enum.

**Solution in one sentence.** Use `auto()` to generate the primary key and `__new__` to attach additional fields; validate those fields separately.

---

# Problem 05 — The container trap: an unresolved auto object

**The situation.** A developer wraps `auto()` inside a list, expecting it to be replaced with a numeric member value.

**Your challenge.** Demonstrate the actual behavior and redesign the declaration to avoid relying on an unresolved `auto` object.

**Before running anything, predict:** Will `[auto(), "tag"]` resolve the inner value in the same way as `auto(), "tag"`?

We will build the solution a little at a time, inspect the surprising behavior, and finish with executable checks.

### Step 1 — Inspect a nested list deliberately

In [14]:
class NestedAuto(Enum):
    WRONG = [auto(), 'tag']

print('Stored value:', NestedAuto.WRONG.value)
print('Type of first item:', type(NestedAuto.WRONG.value[0]).__name__)
assert isinstance(NestedAuto.WRONG.value[0], enum.auto)

Stored value: [auto(_auto_null), 'tag']
Type of first item: auto


**What did we learn?** `auto()` is resolved at the **top level** of an enum member assignment (including a top-level tuple). Nested containers such as this list retain the `auto` instance instead of a generated integer.

### Step 2 — Move the automatic value to the supported location

If we intend to store a tuple itself as `.value`, a custom generator depending only on the declaration index is straightforward. Here we do not need a custom `__new__`: the tuple becomes the member value, in contrast with Problem 4.

In [15]:
class TaggedValue(Enum):
    def _generate_next_value_(name, start, count, last_values):
        return 1 + count

    FIRST = auto(), 'alpha'
    SECOND = auto(), 'beta'

print(TaggedValue.FIRST.value, TaggedValue.SECOND.value)
assert TaggedValue.FIRST.value == (1, 'alpha')
assert TaggedValue.SECOND.value == (2, 'beta')

(1, 'alpha') (2, 'beta')


### Step 3 — Compare the two good representations

**Two designs, two meanings.** `TaggedValue.FIRST.value` is a tuple `(1, 'alpha')`. By contrast, `Route.SEARCH.value` is an integer and its path is `.path` because a custom `__new__` unpacked those fields. Choose deliberately based on your intended lookup and serialization API.

In [16]:
assert isinstance(TaggedValue.FIRST.value, tuple)
assert isinstance(Route.SEARCH.value, int)
assert TaggedValue((2, 'beta')) is TaggedValue.SECOND
assert Route(301) is Route.SEARCH
print('Problem 05: both explicit representations validated')

Problem 05: both explicit representations validated


**Solution in one sentence.** Place `auto()` where the enum machinery supports it, and decide whether metadata belongs in `.value` or in separate attributes.

---

# Problem 06 — Why declaration order can break the generator

**The situation.** A team adds `_generate_next_value_` at the bottom of an enum after introducing its first member.

**Your challenge.** Reproduce the failure safely, correct the order, and explain why this rule follows from class-definition-time generation.

**Before running anything, predict:** Can the generator be changed after the first `auto()` has already been processed?

We will build the solution a little at a time, inspect the surprising behavior, and finish with executable checks.

### Step 1 — Catch the error instead of breaking Run All

We use `exec` only as a small sandbox for an intentionally invalid *fixed literal* class definition. This is not a recommended general application pattern; it is a teaching aid so that the notebook stays runnable.

In [17]:
bad_class_source = '''
from enum import Enum, auto
class LatePolicy(Enum):
    FIRST = auto()
    def _generate_next_value_(name, start, count, last_values):
        return 900 + count
    SECOND = auto()
'''

try:
    exec(bad_class_source, {})
except TypeError as exc:
    print('Expected class-definition error:', exc)
else:
    raise AssertionError('The late generator definition should have failed')

Expected class-definition error: _generate_next_value_ must be defined before members


### Step 2 — Fix the design, not the exception handler

In [18]:
class EarlyPolicy(Enum):
    def _generate_next_value_(name, start, count, last_values):
        return 900 + count

    FIRST = auto()
    SECOND = auto()

assert [item.value for item in EarlyPolicy] == [900, 901]
print(list(EarlyPolicy))

[<EarlyPolicy.FIRST: 900>, <EarlyPolicy.SECOND: 901>]


**What did we learn?** The generator is needed while enum members are being created. In ordinary enum class bodies it must therefore be declared **before the first member**.

### Step 3 — Make the class-creation rule part of code review

A useful review checklist is: (1) define generation policy first, (2) declare members, (3) decide whether aliases are allowed, and (4) run small tests. A class that fails to define cannot be “fixed” by catching the error and continuing with missing members.

In [19]:
assert EarlyPolicy.FIRST.value == 900
assert EarlyPolicy.SECOND.value == 901
print('Problem 06: corrected definition passes')

Problem 06: corrected definition passes


**Solution in one sentence.** Put `_generate_next_value_` before any enum members: definition order is part of the mechanism.

---

# Problem 07 — One generator policy, two independent domains

**The situation.** An application has user actions and audit actions, both of which need a predictable lowercase `v2:` identifier.

**Your challenge.** Create a member-free base enum, reuse its generator twice, and prove that the resulting classes remain distinct types.

**Before running anything, predict:** Does sharing a generator make `UserAction.LOGIN` equal to `AuditAction.LOGIN`?

We will build the solution a little at a time, inspect the surprising behavior, and finish with executable checks.

### Step 1 — Create a reusable base without members

In [20]:
class VersionedNames(Enum):
    def _generate_next_value_(name, start, count, last_values):
        return 'v2:' + name.lower()

class UserAction(VersionedNames):
    LOGIN = auto()
    LOGOUT = auto()

class AuditAction(VersionedNames):
    LOGIN = auto()
    EXPORT = auto()

print([item.value for item in UserAction])
print([item.value for item in AuditAction])

['v2:login', 'v2:logout']
['v2:login', 'v2:export']


### Step 2 — Prove that identical raw values do not identify an enum type

In [21]:
assert UserAction.LOGIN.value == AuditAction.LOGIN.value
assert UserAction.LOGIN is not AuditAction.LOGIN
assert UserAction.LOGIN != AuditAction.LOGIN
assert UserAction('v2:login') is UserAction.LOGIN
assert AuditAction('v2:login') is AuditAction.LOGIN
print('Shared format, independent enum types')

Shared format, independent enum types


**What did we learn?** Enum identity includes the class. Two enums can safely reuse the same generated strings when the domain/type is explicit. If values are stored without any domain discriminator, however, the same string is ambiguous.

### Step 3 — Do not extend a base that already contains members

In [22]:
try:
    class ExtraUserAction(UserAction):
        RESET = auto()
except TypeError as exc:
    print('Expected subclassing error:', exc)
else:
    raise AssertionError('Enums with existing members cannot be extended this way')

Expected subclassing error: <enum 'ExtraUserAction'> cannot extend <enum 'UserAction'>


**Checkpoint / reasoning.** `VersionedNames` had no members, so it can be subclassed to create independent member-bearing enums. `UserAction` does have members, so deriving more members from it is not supported. A reusable base should contain behavior but no members.

**Solution in one sentence.** Share a generator-only parent to standardize formatting; do not confuse equal raw values across distinct enum types.

---

# Problem 08 — Deterministic name-derived IDs

**The situation.** Generated sequential values shift when members are reordered. A local cache needs identifiers that stay the same under reordering.

**Your challenge.** Derive fixed-width IDs from names, reject collisions within each enum, and compare two declaration orders.

**Before running anything, predict:** If you reorder declarations but keep names unchanged, will the numeric IDs stay the same?

We will build the solution a little at a time, inspect the surprising behavior, and finish with executable checks.

### Step 1 — Choose a deterministic mapping, not Python built-in hash()

Python's built-in `hash(str)` is process-randomized and unsuitable for persisted identifiers. Here we use a fixed BLAKE2s digest to obtain a reproducible *example* 64-bit ID. A 64-bit hash can still collide: we detect collisions among declared values but cannot promise they are impossible. Never confuse this example with a cryptographic authentication or access-control scheme.

In [23]:
def stable_id(name):
    raw = hashlib.blake2s(name.encode('utf-8'), digest_size=8,
                          person=b'enum-v2').digest()
    return int.from_bytes(raw, 'big')

print('RECEIVE:', stable_id('RECEIVE'))
assert stable_id('RECEIVE') == stable_id('RECEIVE')

RECEIVE: 3819331091798714159


### Step 2 — Let the generator enforce local uniqueness

In [24]:
class HashNamed(Enum):
    def _generate_next_value_(name, start, count, last_values):
        candidate = stable_id(name)
        if candidate in last_values:
            raise ValueError(f'ID collision for {name}')
        return candidate

class RequestOrderA(HashNamed):
    RECEIVE = auto()
    VALIDATE = auto()
    SEND = auto()

class RequestOrderB(HashNamed):
    SEND = auto()
    RECEIVE = auto()
    VALIDATE = auto()

print({m.name: m.value for m in RequestOrderA})

{'RECEIVE': 3819331091798714159, 'VALIDATE': 6387930764274954460, 'SEND': 17885612520791642612}


### Step 3 — Demonstrate the benefit and its limits

In [25]:
ids_a = {m.name: m.value for m in RequestOrderA}
ids_b = {m.name: m.value for m in RequestOrderB}
assert ids_a == ids_b
assert len(set(ids_a.values())) == len(ids_a)
assert stable_id('SEND') != stable_id('SEND_RENAMED')
print('IDs unchanged under reordering; renaming changes an ID')

IDs unchanged under reordering; renaming changes an ID


**Design tradeoff.** This solves reordering drift, but **not** renaming, collisions with independently generated namespaces, or changing the algorithm/digest size. If an ID is a permanent external protocol contract, explicit assigned identifiers plus a migration plan are usually easier to govern.

**Solution in one sentence.** Deterministic name-derived values can survive reordering; audit collisions and document how renaming changes identity.

---

# Problem 09 — Tolerant input parsing without corrupting enum values

**The situation.** A command-line interface accepts `turbo`, `TURBO`, and an old spelling `quick` for the same operating mode.

**Your challenge.** Keep unique automatic values and implement input normalization via `_missing_` instead of adding unwanted enum members.

**Before running anything, predict:** Will accepting another input spelling add another entry to `Mode.__members__`?

We will build the solution a little at a time, inspect the surprising behavior, and finish with executable checks.

### Step 1 — Generate the canonical wire spellings

In [26]:
@unique
class Mode(Enum):
    def _generate_next_value_(name, start, count, last_values):
        return name.lower().replace('_', '-')

    TURBO = auto()
    ECO = auto()
    BALANCED = auto()

    @classmethod
    def _missing_(cls, value):
        if not isinstance(value, str):
            return None
        key = value.strip().casefold().replace('_', '-')
        old_names = {'quick': 'TURBO', 'slow': 'ECO'}
        if key in old_names:
            return cls[old_names[key]]
        for member in cls:
            if member.value == key:
                return member
        return None

print([(m.name, m.value) for m in Mode])

[('TURBO', 'turbo'), ('ECO', 'eco'), ('BALANCED', 'balanced')]


### Step 2 — Exercise canonical, case-insensitive, and legacy inputs

In [27]:
samples = {'turbo': Mode.TURBO, ' TURBO ': Mode.TURBO,
           'quick': Mode.TURBO, 'SLOW': Mode.ECO,
           'balanced': Mode.BALANCED}
for raw, expected in samples.items():
    assert Mode(raw) is expected
    print(repr(raw), '->', Mode(raw).name)

'turbo' -> TURBO
' TURBO ' -> TURBO
'quick' -> TURBO
'SLOW' -> ECO
'balanced' -> BALANCED


### Step 3 — Reject input that the specification does not accept

In [28]:
for bad in (None, 12, 'warp-speed', ''):
    try:
        Mode(bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f'Unexpectedly accepted {bad!r}')
assert len(Mode.__members__) == 3
assert len(list(Mode)) == 3
print('Invalid inputs rejected; no extra aliases added')

Invalid inputs rejected; no extra aliases added


**Checkpoint / reasoning.** `_missing_` runs only when normal value lookup fails. It maps inputs to an **existing** enum member, not to a new member. Use it for carefully documented backward compatibility; unrestricted fuzzy matching can hide user mistakes.

**Solution in one sentence.** Keep enum identity canonical and put controlled alternate-input handling in `_missing_`.

---

# Problem 10 — Sparse flags with reserved extension slots

**The situation.** A binary message has a two-bit slot for each permission. Only the first bit of a slot is assigned now; the other bit is reserved.

**Your challenge.** Override automatic flag generation, combine permissions, and prove that reserved bits are rejected.

**Before running anything, predict:** How do `READ | WRITE` and an explicitly named composite behave during iteration?

We will build the solution a little at a time, inspect the surprising behavior, and finish with executable checks.

### Step 1 — Compare the documented default with our custom policy

By default, `Flag`/`IntFlag` automatic values use powers of two. Here the protocol demands **every other bit**: 1, 4, 16, …, leaving 2, 8, 32, … unassigned. We choose a generator based on the declaration index and keep composite declarations after the simple bits.

In [29]:
class Permission(Flag):
    def _generate_next_value_(name, start, count, last_values):
        return 1 << (2 * count)

    READ = auto()
    WRITE = auto()
    EXECUTE = auto()
    READ_WRITE = READ | WRITE

print([(m.name, m.value) for m in Permission])
print('All declared names:', list(Permission.__members__))

[('READ', 1), ('WRITE', 4), ('EXECUTE', 16)]
All declared names: ['READ', 'WRITE', 'EXECUTE', 'READ_WRITE']


### Step 2 — Understand flags as combinable bits

In [30]:
assert [m.value for m in Permission] == [1, 4, 16]
assert Permission.READ_WRITE.value == 5
assert Permission.READ | Permission.WRITE is Permission.READ_WRITE
assert Permission.READ in Permission.READ_WRITE
assert Permission.EXECUTE not in Permission.READ_WRITE
print('Composite value:', Permission.READ_WRITE.value)

Composite value: 5


**What did we learn?** `READ_WRITE` names a combination rather than an additional single-bit permission. Iteration over a `Flag` normally shows the named single-bit flags; `__members__` also lists the named composite.

### Step 3 — Test the protocol boundary

In [31]:
for reserved_bit in (2, 8, 32):
    try:
        Permission(reserved_bit)
    except ValueError:
        pass
    else:
        raise AssertionError('Reserved bit was accidentally accepted')
assert Permission(5) is Permission.READ_WRITE
print('Reserved bits rejected by default Flag boundary')

Reserved bits rejected by default Flag boundary


**Evolution warning.** A generator based on `count` assumes that simple members are declared first. If you insert a composite in the middle, you consume a declaration slot and shift later generated positions. Pin bit assignments explicitly once they become part of a published protocol.

**Solution in one sentence.** Automatic flag generation can reserve extension space, but declared order and protocol compatibility still need tests.

---

# Problem 11 — A structured value can accidentally become an alias

**The situation.** A telemetry enum stores immutable `code` and `label` fields in a dataclass value. Two entries are constructed separately but contain identical data.

**Your challenge.** Observe equality-based aliasing and decide whether the data class contains enough information to distinguish events.

**Before running anything, predict:** Does creating two distinct dataclass objects always yield two enum members?

We will build the solution a little at a time, inspect the surprising behavior, and finish with executable checks.

### Step 1 — Build an immutable value type

In [32]:
@dataclass(frozen=True)
class EventSpec:
    code: int
    label: str

@unique
class Telemetry(Enum):
    def _generate_next_value_(name, start, count, last_values):
        return EventSpec(700 + count, name.lower())

    READY = auto()
    STARTED = auto()
    FINISHED = auto()

print([m.value for m in Telemetry])

[EventSpec(code=700, label='ready'), EventSpec(code=701, label='started'), EventSpec(code=702, label='finished')]


### Step 2 — Show that equality, not object identity, determines aliases

The following separate class is intentionally **not** decorated with `@unique`, because its second declared value is structurally equal to the first. Its objects are distinct at construction time, but dataclass equality compares fields.

In [33]:
class DuplicateSpec(Enum):
    READY = EventSpec(700, 'ready')
    READY_AGAIN = EventSpec(700, 'ready')
    FINISHED = EventSpec(701, 'finished')

assert DuplicateSpec.READY_AGAIN is DuplicateSpec.READY
assert len(DuplicateSpec.__members__) == 3
assert len(list(DuplicateSpec)) == 2
print({k: v.name for k, v in DuplicateSpec.__members__.items()})

{'READY': 'READY', 'READY_AGAIN': 'READY', 'FINISHED': 'FINISHED'}


### Step 3 — Make the uniqueness policy explicit

In [34]:
try:
    unique(DuplicateSpec)
except ValueError as exc:
    print('Expected duplicate-spec rejection:', exc)
else:
    raise AssertionError('DuplicateSpec must contain an alias')

assert len({m.value for m in Telemetry}) == len(list(Telemetry))
print('Telemetry has unique immutable values')

Expected duplicate-spec rejection: duplicate values found in <enum 'DuplicateSpec'>: READY_AGAIN -> READY
Telemetry has unique immutable values


**Checkpoint / reasoning.** The fields and their equality semantics define whether two values collide. If a structured value is used, decide which fields belong to the real identity and use an audit or `@unique` as appropriate.

**Solution in one sentence.** Separate object allocations do not guarantee distinct enum members: *value equality* controls aliasing.

---

# Problem 12 — Exhaustion must fail at class definition

**The situation.** A compact hardware register allows only three automatically allocated values. A fourth declaration is a programming error.

**Your challenge.** Make the generator fail predictably, demonstrate the failed definition safely, and confirm that no partial enum leaks into the destination namespace.

**Before running anything, predict:** What should happen if somebody adds a fourth member next month?

We will build the solution a little at a time, inspect the surprising behavior, and finish with executable checks.

### Step 1 — Define a bounded policy as ordinary reusable behavior

In [35]:
class TinyRegistry(Enum):
    def _generate_next_value_(name, start, count, last_values):
        if count >= 3:
            raise ValueError(f'capacity 3 exceeded by {name}')
        return count + 1

class ThreeRegisters(TinyRegistry):
    INPUT = auto()
    OUTPUT = auto()
    CONTROL = auto()

assert [m.value for m in ThreeRegisters] == [1, 2, 3]
print('Within capacity:', list(ThreeRegisters))

Within capacity: [<ThreeRegisters.INPUT: 1>, <ThreeRegisters.OUTPUT: 2>, <ThreeRegisters.CONTROL: 3>]


### Step 2 — Try an invalid fourth member in a separate namespace

A generator exception aborts the class statement. By isolating the deliberately failing example in a dictionary, we can check that the expected class name was never assigned. The bounded policy is better than looping forever or silently creating a duplicate code.

In [36]:
namespace = {'TinyRegistry': TinyRegistry, 'auto': auto}
source = '''
class FourRegisters(TinyRegistry):
    INPUT = auto()
    OUTPUT = auto()
    CONTROL = auto()
    EXTRA = auto()
'''
try:
    exec(source, namespace)
except ValueError as exc:
    assert 'EXTRA' in str(exc)
    print('Expected capacity failure:', exc)
else:
    raise AssertionError('Exhaustion should have raised')
assert 'FourRegisters' not in namespace

Expected capacity failure: capacity 3 exceeded by EXTRA


### Step 3 — Explain why strict failure is preferable

An enum definition is a schema definition. If there are no safe values left, continuing would create ambiguous or out-of-contract members. In production, consider explicit reserved slots, a schema-version upgrade, or widening the field—instead of disguising exhaustion with a random retry loop.

In [37]:
assert len(ThreeRegisters.__members__) == 3
assert [m.value for m in ThreeRegisters] == [1, 2, 3]
print('Problem 12: valid schema intact; invalid schema rejected')

Problem 12: valid schema intact; invalid schema rejected


**Solution in one sentence.** A generator should raise an informative exception when its value space is exhausted; never rely on infinite retries.

---

# Problem 13 — Rename an enum member without losing stored records

**The situation.** Version 1 of an app persisted names such as `PENDING` and `DONE`. Version 2 renames `PENDING` to `QUEUED`.

**Your challenge.** Build explicit name-based migration and prove that both old and new records decode into the version-2 enum.

**Before running anything, predict:** Will automatically generated integer values remain unchanged just because two member names have similar meanings?

We will build the solution a little at a time, inspect the surprising behavior, and finish with executable checks.

### Step 1 — Model the two historical schemas independently

This is a different concern from `_missing_`, which translates *values*. When stored records contain **member names**, use a documented name migration. We deliberately model old and new enums as distinct classes to prevent a false assumption that members from different schema versions are the same objects.

In [38]:
class TaskV1(Enum):
    PENDING = auto()
    RUNNING = auto()
    DONE = auto()

class TaskV2(Enum):
    QUEUED = auto()
    RUNNING = auto()
    DONE = auto()
    CANCELLED = auto()

old_document = {'schema': 1, 'state': 'PENDING'}
print('Old record:', old_document)
print('New names:', list(TaskV2.__members__))

Old record: {'schema': 1, 'state': 'PENDING'}
New names: ['QUEUED', 'RUNNING', 'DONE', 'CANCELLED']


### Step 2 — Write the migration as a small, reviewable mapping

In [39]:
V1_TO_V2 = {'PENDING': 'QUEUED',
            'RUNNING': 'RUNNING',
            'DONE': 'DONE'}

def decode_task(record):
    if not isinstance(record, dict):
        raise TypeError('record must be a mapping')
    version = record.get('schema')
    name = record.get('state')
    if not isinstance(name, str):
        raise ValueError('state must be a member name string')
    if version == 1:
        if name not in V1_TO_V2:
            raise ValueError(f'unknown v1 state: {name}')
        name = V1_TO_V2[name]
    elif version != 2:
        raise ValueError(f'unsupported schema version: {version}')
    try:
        return TaskV2[name]
    except KeyError as exc:
        raise ValueError(f'unknown v2 state: {name}') from exc

assert decode_task(old_document) is TaskV2.QUEUED
assert decode_task({'schema': 2, 'state': 'CANCELLED'}) is TaskV2.CANCELLED

### Step 3 — Create a canonical writer

In [40]:
def encode_task(task):
    if not isinstance(task, TaskV2):
        raise TypeError('expected TaskV2')
    return {'schema': 2, 'state': task.name}

for state in TaskV2:
    assert decode_task(encode_task(state)) is state
print('Current records round-trip correctly:', json.dumps(encode_task(TaskV2.QUEUED)))

Current records round-trip correctly: {"schema": 2, "state": "QUEUED"}


### Step 4 — Check hostile or unsupported records

In [41]:
for bad_record in (
    {'schema': 1, 'state': 'CANCELLED'},
    {'schema': 3, 'state': 'DONE'},
    {'schema': 2, 'state': 'NOT_A_STATE'},
    {'schema': 2, 'state': 123},
):
    try:
        decode_task(bad_record)
    except ValueError:
        pass
    else:
        raise AssertionError(f'Accepted an invalid record: {bad_record}')
print('Unsupported records rejected')

Unsupported records rejected


**Checkpoint / reasoning.** Name-based persistence avoids tying data to automatically generated integer positions, but renames are still compatibility changes. Handle them with an explicit version and migration mapping. The newly added `CANCELLED` has no valid version-1 representation.

**Solution in one sentence.** Persist names with a schema version and migrate renames deliberately; never assume `auto()` integers define a stable storage contract.

---

# Problem 14 — Capstone: an auditable mini event protocol

**The situation.** We need one protocol that supports ergonomic internal members, unique human-readable labels, reserved codes, a legacy input spelling, and a strict JSON boundary.

**Your challenge.** Build the enum in stages, verify aliases and external IDs, then serialize and parse events without exposing the enum generator as an unstable wire contract.

**Before running anything, predict:** Which properties require custom tests beyond simply decorating the enum with `@unique`?

We will build the solution a little at a time, inspect the surprising behavior, and finish with executable checks.

### Step 1 — Write the contract and decide what is automatic

**Protocol contract.** On the wire we publish fixed numeric codes: 200 for `OPEN`, 210 for `PAUSE`, and 220 for `CLOSE`. We can use `auto()` for human-readable internal labels, but we must **not** infer wire codes from their declaration positions. A separate mapping pins the external codes. The protocol additionally accepts the historical *input* spelling `halt` as `PAUSE`, but does not expose it as another enum member.

### Step 2 — Define the internal enum and validate its labels

In [42]:
@unique
class ProtocolEvent(Enum):
    def _generate_next_value_(name, start, count, last_values):
        return name.lower()

    OPEN = auto()
    PAUSE = auto()
    CLOSE = auto()

WIRE_CODES = {
    ProtocolEvent.OPEN: 200,
    ProtocolEvent.PAUSE: 210,
    ProtocolEvent.CLOSE: 220,
}
assert len(WIRE_CODES) == len(list(ProtocolEvent))
assert len(set(WIRE_CODES.values())) == len(WIRE_CODES)
print('Internal labels:', [m.value for m in ProtocolEvent])
print('External codes:', WIRE_CODES)

Internal labels: ['open', 'pause', 'close']
External codes: {<ProtocolEvent.OPEN: 'open'>: 200, <ProtocolEvent.PAUSE: 'pause'>: 210, <ProtocolEvent.CLOSE: 'close'>: 220}


### Step 3 — Implement a strict encoder

The encoder should reject unexpected objects rather than silently turning `str(something)` into a label. A valid event has a fixed schema version, a known numeric code, and a human-readable name. By adding an explicit schema version, we make future migrations possible.

In [43]:
def encode_event(event):
    if not isinstance(event, ProtocolEvent):
        raise TypeError('expected ProtocolEvent')
    return json.dumps({
        'version': 1,
        'code': WIRE_CODES[event],
        'name': event.name,
    }, sort_keys=True)

encoded = encode_event(ProtocolEvent.PAUSE)
print(encoded)
assert json.loads(encoded) == {'code': 210, 'name': 'PAUSE', 'version': 1}

{"code": 210, "name": "PAUSE", "version": 1}


### Step 4 — Build lookup tables once and validate the published mapping

In [44]:
BY_WIRE_CODE = {wire: member for member, wire in WIRE_CODES.items()}

assert set(WIRE_CODES) == set(ProtocolEvent)
assert set(BY_WIRE_CODE) == {200, 210, 220}
assert len(BY_WIRE_CODE) == len(WIRE_CODES)
print('Wire-code table is complete and invertible')

Wire-code table is complete and invertible


### Step 5 — Implement a strict decoder with a documented legacy spelling

The decoder handles untrusted JSON at the boundary. Boolean values deserve special care: `bool` subclasses `int` in Python, so `isinstance(True, int)` is true. We require `type(code) is int` to reject a boolean where the wire schema demands an integer.

In [45]:
def decode_event(blob):
    try:
        document = json.loads(blob)
    except (TypeError, ValueError) as exc:
        raise ValueError('invalid JSON event') from exc
    if not isinstance(document, dict):
        raise ValueError('event must be a JSON object')
    if set(document) != {'version', 'code', 'name'}:
        raise ValueError('wrong event fields')
    if type(document['version']) is not int or document['version'] != 1:
        raise ValueError('unsupported version')
    code_value = document['code']
    if type(code_value) is not int or code_value not in BY_WIRE_CODE:
        raise ValueError('invalid event code')
    event = BY_WIRE_CODE[code_value]
    name = document['name']
    if not isinstance(name, str):
        raise ValueError('event name must be a string')
    mapped_name = {'halt': 'PAUSE'}.get(name, name)
    if mapped_name != event.name:
        raise ValueError('event code/name mismatch')
    return event

assert decode_event(encoded) is ProtocolEvent.PAUSE

### Step 6 — Walk through a legitimate legacy record

In [46]:
legacy_event = json.dumps({'version': 1, 'code': 210, 'name': 'halt'})
assert decode_event(legacy_event) is ProtocolEvent.PAUSE
assert encode_event(decode_event(legacy_event)) == encode_event(ProtocolEvent.PAUSE)
print('Legacy input normalized to canonical output:', encode_event(decode_event(legacy_event)))

Legacy input normalized to canonical output: {"code": 210, "name": "PAUSE", "version": 1}


**What did we learn?** The old spelling exists only at the decoder boundary. The canonical writer always produces `PAUSE`. We did not introduce a shadow enum member, a duplicate numeric wire code, or an unstable automatic wire value.

### Step 7 — Try adversarial records

In [47]:
invalid_blobs = [
    'not JSON',
    '[]',
    json.dumps({'version': 1, 'code': True, 'name': 'OPEN'}),
    json.dumps({'version': 1, 'code': 999, 'name': 'OPEN'}),
    json.dumps({'version': 1, 'code': 200, 'name': 'CLOSE'}),
    json.dumps({'version': 2, 'code': 200, 'name': 'OPEN'}),
    json.dumps({'version': 1, 'code': 200, 'name': 'OPEN', 'extra': 1}),
    json.dumps({'version': 1, 'code': 200, 'name': None}),
]
for bad_blob in invalid_blobs:
    try:
        decode_event(bad_blob)
    except ValueError:
        pass
    else:
        raise AssertionError(f'Accepted invalid event: {bad_blob}')
print(f'All {len(invalid_blobs)} invalid events rejected')

All 8 invalid events rejected


### Step 8 — Finish with a full round-trip test

In [48]:
for event in ProtocolEvent:
    assert decode_event(encode_event(event)) is event
assert len(list(ProtocolEvent)) == 3
assert len(ProtocolEvent.__members__) == 3
print('Problem 14: canonical round trips and alias audit passed')

Problem 14: canonical round trips and alias audit passed


**Capstone discussion.** `@unique` checks duplicate enum member values. It does not prove that the separate wire-code dictionary is complete, that its numeric codes are unique, that a JSON record has exactly the expected fields, or that `code` and `name` agree. Each of those is a different contract and needs its own test.

**Solution in one sentence.** Use automatic values for convenient internal representation, explicitly pin external protocol identifiers, and validate both directions of the boundary.

---

# Final review: test the concepts together

This last cell is intentionally small. Each earlier problem contains its own runnable assertions; these final checks make sure key independent promises still hold after all cells run.

**Try answering before running:** why can the `JobStage` enum have four names but only two iterated members, while the capstone must have exactly three of each?

In [49]:
assert len(JobStage.__members__) == 4 and len(list(JobStage)) == 2
assert len(ProtocolEvent.__members__) == len(list(ProtocolEvent)) == 3
assert len({m.value for m in Route}) == 3
assert ids_a == ids_b
assert all(decode_task(encode_task(m)) is m for m in TaskV2)
assert all(decode_event(encode_event(m)) is m for m in ProtocolEvent)
print('FINAL AUDIT: 6 independent checks passed')

FINAL AUDIT: 6 independent checks passed


# What to remember

- `auto()` calls `_generate_next_value_` **during class creation**. Its four arguments describe the current declaration context; custom generators should encode a clear invariant.
- Aliasing depends on member **value equality**. Iteration normally suppresses aliases; `__members__` retains declared names.
- Put the custom generator before any member. Share behavior in a member-free base enum when several enums need the same policy.
- A top-level tuple can resolve `auto()`; an `auto()` buried in a list remains unresolved. Custom `__new__` controls whether tuple metadata is part of `.value` or exposed separately.
- Generated integers can shift after edits. Deterministic name hashes trade order dependence for rename/collision concerns. For externally published numeric protocols, pin codes explicitly.
- Use purpose-specific tests: uniqueness, reserved values, field validation, data migration, and round trips are separate properties.

## Additional practice — without supplied code

1. Extend the reserved-code allocator with a **finite list of free codes** and specify its behavior if a manual assignment consumes one.
2. Add a third old spelling to the `_missing_` parser and prove that it does not create another member.
3. Expand the flag class with a fourth permission, then show why moving the composite declaration before it changes a count-based generator.
4. Refactor the capstone to support `version=2` with a new fixed code and a migration rule for old records.
5. Write a static analyzer that reports alias names from `__members__` for every enum defined in this notebook.

## References

- Tutorial supplied by the user: *Automatic Values* (original lesson and progression).
- [Python enum library documentation](https://docs.python.org/3/library/enum.html).
- [Python Enum HOWTO](https://docs.python.org/3/howto/enum.html).

**End of Volume II.**